In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section {{SECTION_NUM}}
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    {{TITLE}}<br>
    <span style="color:#63b3ed;">{{SUBTITLE}}</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    {{SUBTITLE}}
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    {{TAGS}}
  </div>
</div>
"""))


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Environment Check</h2>
</div>

First — verify the environment is ready.

In [ ]:
!python3 ../../scripts/setup_check.py


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: Server Discovery &amp; Warmup</h2>
</div>

Discover running MLX servers and warm them up in parallel.

In [ ]:
# Discover running MLX servers and warm up in parallel
from openai import OpenAI
import time
import concurrent.futures
from IPython.display import HTML, display

# Import shared helpers (includes discover_servers)
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../scripts").resolve()))
import notebook_helpers

# Discover servers (reads model ID from process command line, not /v1/models)
print("Discovering MLX servers...", flush=True)
MODELS, clients = notebook_helpers.discover_servers()

if not MODELS:
    raise RuntimeError("No MLX servers found! Start at least one on ports 8800-8802.")

for m in MODELS:
    print(f"  Port {m['port']}: {m['label']} ({m['model']})", flush=True)

# Warm up all discovered models in parallel
def warmup(m):
    t0 = time.time()
    try:
        clients[m["label"]].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=1,
        )
        return m["label"], time.time() - t0, True
    except Exception as e:
        return m["label"], time.time() - t0, False

print(f"\nWarming up {len(MODELS)} models in parallel...", flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    warmup_results = list(ex.map(warmup, MODELS))

# Display status table
rows = ""
for label, elapsed, ok in warmup_results:
    status = "\u2713" if ok else "\u2717"
    color = "#16a34a" if ok else "#dc2626"
    m = next(m for m in MODELS if m["label"] == label)
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{m['port']}</td>"
        f"<td style='padding:6px 12px; color:#374151; font-size:0.85em; border-bottom:1px solid #e5e7eb;'>{m['model']}</td>"
        f"<td style='padding:6px 12px; color:{color}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{status}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{elapsed:.1f}s</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Port</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Model ID</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Status</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Warmup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))
print(f"All {len(MODELS)} models ready!")

notebook_helpers.init(MODELS, clients)


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧰 Step 3: Load Helpers</h2>
</div>

Import shared utility functions.

In [ ]:
from notebook_helpers import strip_think, compare_models, show_metrics_table
print("Helpers loaded.")


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧪 Step 4: Example Inference</h2>
</div>

Run a prompt across all models and compare results.

In [ ]:
results = compare_models("Explain quantum computing in 2 sentences.")
show_metrics_table(results)
